# Atelier Préparation de Données Textuelles

**Contexte**


Une entreprise souhaite développer un système capable de classer automatiquement des avis
clients afin d'identifier leur sentiment.
Les avis proviennent de plusieurs sources : site web, application mobile, formulaire de satisfaction,
réseaux sociaux et service client.
Les données collectées sont cependant de qualité variable : textes vides, doublons, fautes de frappe,
majuscules/minuscules, caractères spéciaux, emojis, URLs, mentions, répétitions de caractères,
textes très courts ou très longs, plusieurs catégories de sentiment et quelques valeurs manquantes.
Avant de construire un modèle de Machine Learning ou de Deep Learning, les apprenants doivent
donc construire un pipeline complet de préparation des données textuelles.

**Objectifs pédagogiques**

À la fin de l'atelier, l'apprenant doit être capable de :
1) explorer un corpus textuel ;
2) identifier les problèmes de qualité des données ;
3) nettoyer les textes ;
4) tokeniser les textes ;
5) normaliser les données textuelles ;
6) transformer les textes en représentations numériques ;
7) comparer plusieurs méthodes de vectorisation ;
8) préparer un corpus exploitable par un modèle de ML/DL.

**Structure du projet :**
```
atelier_prepa_donnees_textuelles/
│
├── notebooks/
│   └── atelier_prepa_donnees_textuelles.ipynb
│
└── data/
    ├── smart_reviews_raw.csv        <- données brutes fournies
    ├── smart_reviews_cleaned.csv    <- créé après la Partie 4 (normalisation)
    └── tfidf_matrix.npz             <- créé après la Partie 6 (vectorisation)
```

**Imports des bibliothéques et configuration**

In [1]:
#  pandas / numpy : Pour la manipulation de données tabulaires et calcul numérique
import pandas as pd
import numpy as np

# re : Pour le module Python natif d'expressions régulières (regex), notre outil principal
#     pour repérer et supprimer URLs, mentions, hashtags, ponctuation, etc.
import re

#  matplotlib / seaborn : Pour la visualisation
import matplotlib.pyplot as plt
import seaborn as sns

#  collections.Counter : pour compter facilement des occurrences (mots, tokens...)
from collections import Counter

#  pathlib : Pour la gestion des chemins de fichiers
from pathlib import Path

#  scikit-learn : Pour le découpage train/test et vectorisation (Bag of Words, TF-IDF)
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

#  scipy.sparse : pour sauvegarder/charger une matrice creuse (sparse) comme celle produite
#     par la vectorisation de texte (voir Partie 6)
import scipy.sparse as sp

sns.set_theme(style="whitegrid")
pd.set_option("display.max_colwidth", 120)
RANDOM_STATE = 42

DATA_DIR = Path("../data")
RAW_PATH = DATA_DIR / "smart_reviews_raw.csv"
CLEANED_PATH = DATA_DIR / "smart_reviews_cleaned.csv"
MATRIX_PATH = DATA_DIR / "tfidf_matrix.npz"

print("Configuration prête.")

Configuration prête.


**Partie 1 – Exploration du corpus**

1) Charger les données CSV 

In [2]:
df = pd.read_csv("../data/smart_reviews_raw.csv")

df.head(3)

,id_avis,date,source,produit,texte,sentiment,note,langue
0,AV0001,2026-02-27,mobile,Ordinateur NovaBook,"Très bonne expérience, simple et efficace.",positif,4,fr
1,AV0002,2026-01-09,web,SmartPhone X,Très satisfait de mon achat 👍 #avis,positif,4,fr
2,AV0003,2026-07-03,réseaux_sociaux,Écouteurs AirSound,"Produit parfait, rien à signaler.",positif,4,fr


2) Combien d'avis contient le dataset 

In [3]:
n_avis = df.shape[0]
print(f"Le dataset contient {n_avis} avis.")

Le dataset contient 1200 avis.


3) Combien de colonnes possède-t-il ?

In [5]:
nb_cols = df.shape[1]
print(f"Le dataset contient {nb_cols} colonnes.")

df.columns.tolist()

Le dataset contient 8 colonnes.


['id_avis',
 'date',
 'source',
 'produit',
 'texte',
 'sentiment',
 'note',
 'langue']

4) Quel est le type de chaque colonne ?

In [7]:
df.dtypes

id_avis      object
date         object
source       object
produit      object
texte        object
sentiment    object
note          int64
langue       object
dtype: object

5) Existe-t-il des valeurs manquantes ?

In [9]:
valeurs_manquantes = df.isna().sum()
pourcentage_manquant = (df.isna().mean() * 100).round(2)

pd.DataFrame({
    "nb_manquants": valeurs_manquantes,
    "pct_manquants": pourcentage_manquant,
})

,nb_manquants,pct_manquants
id_avis,0,0.00
date,0,0.00
source,0,0.00
produit,0,0.00
texte,5,0.42
sentiment,0,0.00
note,0,0.00
langue,0,0.00


**Interprétation :** seule la colonne `texte` contient des valeurs manquantes. C'est LA colonne
la plus critique de tout l'atelier (c'est elle qu'on va nettoyer, tokeniser, vectoriser...) : ces
lignes sans texte seront supprimées dès le début de la Partie 2, puisqu'un modèle de classification
de sentiment ne peut rien faire d'un avis vide.

6) Identifier quelques types de texte en affichant par exemple : texte normal ; texte vide ; texte
contenant une URL ; texte contenant une mention ; texte contenant un hashtag ; texte
contenant des emojis ; texte avec beaucoup de ponctuation ; texte en majuscules ; texte avec
répétition de caractères.

In [10]:
def afficher_exemple(masque, titre, colonne="texte"):
    """Affiche le premier exemple de df correspondant au masque booléen donné."""
    sous_ensemble = df[masque]
    print(f"--- {titre} ({len(sous_ensemble)} avis trouvés) ---")
    if len(sous_ensemble) > 0:
        print(repr(sous_ensemble[colonne].iloc[0]))
    else:
        print("(aucun exemple trouvé)")
    print()


texte = df["texte"]

# texte normal : un texte "propre" sans caractère spécial
afficher_exemple(texte.notna() & ~texte.str.contains(r'http|@|#|[👍😊😢😡]', na=False, regex=True),
                  "Texte normal")

# texte vide : valeur manquante (NaN) OU chaîne vide/espaces uniquement
afficher_exemple(texte.isna() | (texte.fillna("").str.strip() == ""), "Texte vide")

# texte contenant une URL
afficher_exemple(texte.str.contains(r'https?://', na=False, regex=True), "Texte avec URL")

# texte contenant une mention (@pseudo)
afficher_exemple(texte.str.contains(r'@\w+', na=False, regex=True), "Texte avec mention")

# texte contenant un hashtag (#mot)
afficher_exemple(texte.str.contains(r'#\w+', na=False, regex=True), "Texte avec hashtag")

# texte contenant des emojis (on cible ici les plages Unicode des emojis les plus courants)
afficher_exemple(texte.str.contains(r'[\U0001F300-\U0001FAFF\U00002600-\U000027BF]', na=False, regex=True),
                  "Texte avec emoji")

# texte avec beaucoup de ponctuation (3 signes de ponctuation identiques ou plus à la suite)
afficher_exemple(texte.str.contains(r'([!?.]){2,}', na=False, regex=True), "Texte avec ponctuation répétée")

# texte en majuscules (au moins un mot de 3 lettres ou plus tout en majuscules)
afficher_exemple(texte.str.contains(r'\b[A-ZÀ-Ü]{3,}\b', na=False, regex=True), "Texte en majuscules")

# texte avec répétition de caractères (ex: "trèèès", "supeeer")
afficher_exemple(texte.str.contains(r'(\w)\1{2,}', na=False, regex=True), "Texte avec répétition de caractères")

--- Texte normal (627 avis trouvés) ---
'Très  bonne  expérience,  simple  et  efficace.'

--- Texte vide (6 avis trouvés) ---
nan

--- Texte avec URL (143 avis trouvés) ---
'Produit parfait, rien à signaler. https://example.com/commande/17'

--- Texte avec mention (128 avis trouvés) ---
'@client Livraison rapide et produit conforme à mes attentes.'

--- Texte avec hashtag (127 avis trouvés) ---
'Très satisfait de mon achat 👍 #avis'

--- Texte avec emoji (192 avis trouvés) ---
'Très satisfait de mon achat 👍 #avis'

--- Texte avec ponctuation répétée (141 avis trouvés) ---
'Produit excellent, je suis très satisfait. !!!'

--- Texte en majuscules (144 avis trouvés) ---
"LA BATTERIE TIENT VRAIMENT BIEN ET L'ÉCRAN EST SUPERBE."

--- Texte avec répétition de caractères (22 avis trouvés) ---
'Produit excellent, je suis trèèès satisfait.'



C:\Users\HP\AppData\Local\Temp\ipykernel_7148\262054705.py:35: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  afficher_exemple(texte.str.contains(r'([!?.]){2,}', na=False, regex=True), "Texte avec ponctuation répétée")
C:\Users\HP\AppData\Local\Temp\ipykernel_7148\262054705.py:41: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  afficher_exemple(texte.str.contains(r'(\w)\1{2,}', na=False, regex=True), "Texte avec répétition de caractères")


7) Mesurer la longueur des textes en créant une nouvelle colonne « longueur » pour déterminer :
longueur minimale ; longueur maximale ; longueur moyenne ; médiane et quartiles.

On crée une colonne `longueur` (nombre de caractères) pour chaque avis, en traitant les valeurs
manquantes comme des textes de longueur 0.

In [11]:
df["longueur"] = df["texte"].fillna("").str.len()

statistiques_longueur = df["longueur"].describe()
print(statistiques_longueur)
print()
print("Médiane :", df["longueur"].median())
print()
print("Quartiles :")
print(df["longueur"].quantile([0.25, 0.5, 0.75]))

count    1200.000000
mean       48.963333
std        22.350913
min         0.000000
25%        37.000000
50%        48.000000
75%        56.000000
max       636.000000
Name: longueur, dtype: float64

Médiane : 48.0

Quartiles :
0.25    37.0
0.50    48.0
0.75    56.0
Name: longueur, dtype: float64
